In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/kanchandalal123/mcq-ranking-train-dataset-2/ranking_train (1).csv


# BERT + Retrieval-Augmented Generation (RAG) Pipeline for Smart MCQ Solver

## Objective

In this notebook we build a Retrieval-Augmented Multiple Choice Question Answering (RAG-MCQA) system.


Models Used:

- SentenceTransformer (Embeddings)
- FAISS (Vector Search)
- BERT-base-uncased (Multiple Choice)

In [2]:

!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install datasets
!pip -q install evaluate
!pip -q install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 68.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [3]:

import os
import gc
import random
import warnings
import pickle

import numpy as np
import pandas as pd

import torch

from datasets import Dataset

from transformers import (

    AutoTokenizer,

    BertTokenizer,

    BertForMultipleChoice,

    Trainer,

    TrainingArguments

)

from sentence_transformers import SentenceTransformer

import faiss

from sklearn.model_selection import train_test_split

from tqdm.auto import tqdm

import wandb

warnings.filterwarnings("ignore")

In [4]:
os.environ["WANDB_API_KEY"] = "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"

wandb.init(

    project="24f1002360-t22026",

    name="bert-rag",

    job_type="training"

)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 24f1002360 (24f1002360-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

In [6]:

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)

print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## knowlegde base 

In [36]:

knowledge_base = []

for _, row in train.iterrows():

    knowledge_base.append({

        "question_id": row["id"],

        "question": row["prompt"],

        "document": row["prompt"]

    })

knowledge_base = pd.DataFrame(knowledge_base)

print("Knowledge Base Size :", knowledge_base.shape)

knowledge_base.head()

Knowledge Base Size : (2000, 3)


,question_id,question,document
0,1,Pick the best possible answer: What is Martin ...,Pick the best possible answer: What is Martin ...
1,2,What is accelerator-based light-ion fusion?,What is accelerator-based light-ion fusion?
2,3,Determine the correct option: What is the term...,Determine the correct option: What is the term...
3,4,Select the most accurate option: What is Marti...,Select the most accurate option: What is Marti...
4,5,Identify the correct statement: What is the co...,Identify the correct statement: What is the co...


In [37]:
SAVE_DIR = "/kaggle/working/rag_data"

os.makedirs(SAVE_DIR, exist_ok=True)

knowledge_base.to_csv(
    f"{SAVE_DIR}/knowledge_base.csv",
    index=False)

print("Knowledge Base Saved")

Knowledge Base Saved


## Embeddings

In [38]:

embedding_model = SentenceTransformer(

    "BAAI/bge-small-en-v1.5"

)

print("Embedding Model Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded


In [39]:
documents = knowledge_base["document"].tolist()

embeddings = embedding_model.encode(

    documents,

    show_progress_bar=True,

    convert_to_numpy=True

)

print(embeddings.shape)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 384)


## FAISS Index

In [40]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors Indexed :", index.ntotal)

Vectors Indexed : 2000


In [41]:
faiss.write_index(
    index,
    f"{SAVE_DIR}/faiss.index"
)

print("FAISS Index Saved")

FAISS Index Saved


## Retrival function 

In [42]:
TOP_K = 5

def retrieve_documents(query, top_k=TOP_K):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k + 1
    )

    retrieved_docs = []

    for idx in indices[0]:

        document = knowledge_base.iloc[idx]["document"]

        # Skip the identical question
        if query.strip() in document:
            continue

        retrieved_docs.append(document)

        if len(retrieved_docs) == top_k:
            break

    return retrieved_docs

In [43]:
sample_question = train.iloc[0]["prompt"]

contexts = retrieve_documents(

    sample_question,

    top_k=3

)

for i, context in enumerate(contexts):

    print("=" * 80)

    print(f"Retrieved Document {i+1}")

    print("=" * 80)

    print(context[:700])

Retrieved Document 1
Select the most accurate option: What is Martin Heidegger's view on the relationship between time and human existence? from the following choices.
Retrieved Document 2
Select the most accurate option: What is Martin Heidegger's view on the relationship between time and human existence? carefully.


In [44]:
wandb.config.update({

    "Embedding Model": "BAAI/bge-small-en-v1.5",

    "Embedding Dimension": embeddings.shape[1],

    "Knowledge Base Size": len(knowledge_base),

    "FAISS Index": "IndexFlatL2"

})

In [45]:

def build_context(query, top_k=TOP_K):

    docs = retrieve_documents(query, top_k)

    return "\n\n".join(docs)

## Bert model 

In [46]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [47]:
ranking_train = pd.read_csv(
    "/kaggle/input/datasets/kanchandalal123/mcq-ranking-train-dataset-2/ranking_train (1).csv"
)

print("Ranking Dataset Shape :", ranking_train.shape)

ranking_train.head()

Ranking Dataset Shape : (10000, 8)


,id,question,option_label,option_text,label,text,text_length,fold
0,1,Pick the best possible answer: What is Martin ...,A,Martin Heidegger believes that humans exist wi...,0,Question: Pick the best possible answer: What ...,474,3
1,1,Pick the best possible answer: What is Martin ...,B,Martin Heidegger believes that humans do not e...,1,Question: Pick the best possible answer: What ...,425,3
2,1,Pick the best possible answer: What is Martin ...,C,Martin Heidegger does not believe in the exist...,0,Question: Pick the best possible answer: What ...,389,3
3,1,Pick the best possible answer: What is Martin ...,D,Martin Heidegger believes that the relationshi...,0,Question: Pick the best possible answer: What ...,369,3
4,1,Pick the best possible answer: What is Martin ...,E,Martin Heidegger believes that time is an illu...,0,Question: Pick the best possible answer: What ...,358,3


In [48]:

context_dict = {}

unique_questions = ranking_train["question"].unique()

for question in tqdm(unique_questions):

    context_dict[question] = build_context(question)

print("Total Contexts :", len(context_dict))

  0%|          | 0/1758 [00:00<?, ?it/s]

Total Contexts : 1758


In [49]:
ranking_train["retrieved_context"] = ranking_train["question"].map(
    context_dict
)

ranking_train.head()

,id,question,option_label,option_text,label,text,text_length,fold,retrieved_context
0,1,Pick the best possible answer: What is Martin ...,A,Martin Heidegger believes that humans exist wi...,0,Question: Pick the best possible answer: What ...,474,3,Select the most accurate option: What is Marti...
1,1,Pick the best possible answer: What is Martin ...,B,Martin Heidegger believes that humans do not e...,1,Question: Pick the best possible answer: What ...,425,3,Select the most accurate option: What is Marti...
2,1,Pick the best possible answer: What is Martin ...,C,Martin Heidegger does not believe in the exist...,0,Question: Pick the best possible answer: What ...,389,3,Select the most accurate option: What is Marti...
3,1,Pick the best possible answer: What is Martin ...,D,Martin Heidegger believes that the relationshi...,0,Question: Pick the best possible answer: What ...,369,3,Select the most accurate option: What is Marti...
4,1,Pick the best possible answer: What is Martin ...,E,Martin Heidegger believes that time is an illu...,0,Question: Pick the best possible answer: What ...,358,3,Select the most accurate option: What is Marti...


In [50]:

ranking_train["text"] = (

    "Retrieved Context: "

    + ranking_train["retrieved_context"]

    + " [SEP] Question: "

    + ranking_train["question"]

    + " [SEP] Option: "

    + ranking_train["option_text"]

)

ranking_train.head()

,id,question,option_label,option_text,label,text,text_length,fold,retrieved_context
0,1,Pick the best possible answer: What is Martin ...,A,Martin Heidegger believes that humans exist wi...,0,Retrieved Context: Select the most accurate op...,474,3,Select the most accurate option: What is Marti...
1,1,Pick the best possible answer: What is Martin ...,B,Martin Heidegger believes that humans do not e...,1,Retrieved Context: Select the most accurate op...,425,3,Select the most accurate option: What is Marti...
2,1,Pick the best possible answer: What is Martin ...,C,Martin Heidegger does not believe in the exist...,0,Retrieved Context: Select the most accurate op...,389,3,Select the most accurate option: What is Marti...
3,1,Pick the best possible answer: What is Martin ...,D,Martin Heidegger believes that the relationshi...,0,Retrieved Context: Select the most accurate op...,369,3,Select the most accurate option: What is Marti...
4,1,Pick the best possible answer: What is Martin ...,E,Martin Heidegger believes that time is an illu...,0,Retrieved Context: Select the most accurate op...,358,3,Select the most accurate option: What is Marti...


In [51]:
FOLD = 0

train_df = (

    ranking_train[ranking_train.fold != FOLD]

    .reset_index(drop=True)

)

valid_df = (

    ranking_train[ranking_train.fold == FOLD]

    .reset_index(drop=True)

)

print("Training Shape :", train_df.shape)

print("Validation Shape :", valid_df.shape)

Training Shape : (8000, 9)
Validation Shape : (2000, 9)


In [57]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

MAX_LENGTH = 384

print("Tokenizer Loaded Successfully!")

Tokenizer Loaded Successfully!


In [58]:
def tokenize(example):

    encoded = tokenizer(

        example["text"],

        truncation=True,

        max_length=MAX_LENGTH

    )

    encoded.pop("token_type_ids", None)

    return encoded

In [62]:

from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label"]]
)

valid_ds = Dataset.from_pandas(
    valid_df[["text", "label"]]
)

print(train_ds)
print(valid_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 8000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})


In [63]:
train_ds = train_ds.map(
    tokenize,
    batched=True
)

valid_ds = valid_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [64]:

train_ds = train_ds.remove_columns(["text"])

valid_ds = valid_ds.remove_columns(["text"])

In [65]:

train_ds.set_format("torch")

valid_ds.set_format("torch")

print(train_ds)

print(valid_ds)

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 8000
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 2000
})


In [66]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "bert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2,

    torch_dtype=torch.float32

)

print(model)

print()

print("Model Loaded Successfully!")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [67]:

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [68]:
valid_metadata = valid_df[
    [
        "id",
        "option_label",
        "label"
    ]
].reset_index(drop=True)

valid_metadata.head()

,id,option_label,label
0,5,A,1
1,5,B,0
2,5,C,0
3,5,D,0
4,5,E,0


In [69]:
def apk(actual, predicted, k=3):

    if actual in predicted[:k]:

        return 1.0 / (

            predicted.index(actual) + 1

        )

    return 0.0


def mapk(actuals, predictions, k=3):

    return np.mean(

        [

            apk(a, p, k)

            for a, p in zip(actuals, predictions)

        ]

    )

In [70]:

from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = torch.softmax(

        torch.tensor(logits),

        dim=1

    )[:,1].numpy()

    temp = valid_metadata.copy()

    temp["score"] = probs

    predictions = []

    actuals = []

    for _, grp in temp.groupby("id"):

        grp = grp.sort_values(

            "score",

            ascending=False

        )

        predictions.append(

            grp["option_label"].tolist()

        )

        actuals.append(

            grp.loc[
                grp["label"] == 1,
                "option_label"
            ].values[0]
        )

    preds = np.argmax(

        logits,

        axis=1

    )

    return {

        "accuracy": accuracy_score(
            labels,
            preds
        ),

        "f1": f1_score(
            labels,
            preds
        ),

        "map3": mapk(
            actuals,
            predictions
        )

    }

In [71]:
from transformers import EarlyStoppingCallback

early_stop = EarlyStoppingCallback(

    early_stopping_patience=2,

    early_stopping_threshold=0.0005

)

In [72]:
training_args = TrainingArguments(

    output_dir="bert_rag_model",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=8,

    weight_decay=0.01,

    warmup_ratio=0.10,

    lr_scheduler_type="cosine",

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=50,

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="map3",

    greater_is_better=True,

    report_to="wandb",

    fp16=False,

    bf16=False,

    seed=SEED

)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [73]:
wandb.config.update({

    "Model": MODEL_NAME,

    "Embedding Model": "BAAI/bge-small-en-v1.5",

    "Retriever": "FAISS",

    "Top K": TOP_K,

    "Max Length": MAX_LENGTH,

    "Epochs": 8,

    "Batch Size": 8,

    "Learning Rate": 2e-5,

    "Weight Decay": 0.01,

    "Scheduler": "Cosine",

    "Knowledge Base Size": len(knowledge_base)

})

In [75]:

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=valid_ds,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[early_stop]

)

In [76]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Map3
1,0.987727,0.998093,0.800000,0.000000,0.572083
2,0.805601,0.705029,0.865000,0.564516,0.866667
3,0.522246,0.370097,0.924500,0.792297,0.953333
4,0.258963,0.277227,0.952000,0.869919,0.977083
5,0.166797,0.171315,0.969000,0.916442,0.985833
6,0.069687,0.116447,0.979000,0.945026,0.991250
7,0.042152,0.097770,0.986500,0.965251,0.991250
8,0.012066,0.091721,0.989500,0.973111,0.991250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=4000, training_loss=0.415374883338809, metrics={'train_runtime': 2317.3458, 'train_samples_per_second': 27.618, 'train_steps_per_second': 1.726, 'total_flos': 9484257323695680.0, 'train_loss': 0.415374883338809, 'epoch': 8.0})

In [77]:

results = trainer.evaluate()

print(results)

{'eval_loss': 0.11642973124980927, 'eval_accuracy': 0.9795, 'eval_f1': 0.9464052287581699, 'eval_map3': 0.99125, 'eval_runtime': 19.7876, 'eval_samples_per_second': 101.074, 'eval_steps_per_second': 6.317, 'epoch': 8.0}


In [78]:
metrics = pd.DataFrame([results])

metrics.to_csv(

    "bert_rag_metrics.csv",

    index=False

)

metrics

,eval_loss,eval_accuracy,eval_f1,eval_map3,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,0.11643,0.9795,0.946405,0.99125,19.7876,101.074,6.317,8.0


In [79]:

SAVE_PATH = "/kaggle/working/bert_rag_model"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!


In [80]:

import gc

gc.collect()

torch.cuda.empty_cache()

print("GPU Memory Cleared!")

GPU Memory Cleared!


## inference and test data preparation 

In [81]:
test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print("Test Shape:", test.shape)

test.head()

Test Shape: (500, 7)


,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [82]:
def create_test_ranking(df):

    rows = []

    for _, row in df.iterrows():

        for option in ["A", "B", "C", "D", "E"]:

            rows.append({

                "id": row["id"],

                "question": row["prompt"],

                "option_label": option,

                "option_text": row[option]

            })

    return pd.DataFrame(rows)

ranking_test = create_test_ranking(test)

print("Ranking Test Shape:", ranking_test.shape)

ranking_test.head()

Ranking Test Shape: (2500, 4)


,id,question,option_label,option_text
0,1,Pick the best possible answer: What is the rel...,A,"For every eigenstate of one Hamiltonian, its p..."
1,1,Pick the best possible answer: What is the rel...,B,"For every eigenstate of one Hamiltonian, its p..."
2,1,Pick the best possible answer: What is the rel...,C,"For every eigenstate of one Hamiltonian, its p..."
3,1,Pick the best possible answer: What is the rel...,D,"For every eigenstate of one Hamiltonian, its p..."
4,1,Pick the best possible answer: What is the rel...,E,"For every eigenstate of one Hamiltonian, its p..."


In [83]:
test_context_dict = {}

unique_test_questions = ranking_test["question"].unique()

for question in tqdm(unique_test_questions):

    test_context_dict[question] = build_context(question)

print("Total Test Contexts:", len(test_context_dict))

  0%|          | 0/492 [00:00<?, ?it/s]

Total Test Contexts: 492


In [84]:
ranking_test["retrieved_context"] = ranking_test["question"].map(

    test_context_dict

)

ranking_test.head()

,id,question,option_label,option_text,retrieved_context
0,1,Pick the best possible answer: What is the rel...,A,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...
1,1,Pick the best possible answer: What is the rel...,B,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...
2,1,Pick the best possible answer: What is the rel...,C,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...
3,1,Pick the best possible answer: What is the rel...,D,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...
4,1,Pick the best possible answer: What is the rel...,E,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...


In [85]:
ranking_test["text"] = (

    "Retrieved Context: "

    + ranking_test["retrieved_context"]

    + " [SEP] Question: "

    + ranking_test["question"]

    + " [SEP] Option: "

    + ranking_test["option_text"]

)

ranking_test.head()

,id,question,option_label,option_text,retrieved_context,text
0,1,Pick the best possible answer: What is the rel...,A,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...
1,1,Pick the best possible answer: What is the rel...,B,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...
2,1,Pick the best possible answer: What is the rel...,C,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...
3,1,Pick the best possible answer: What is the rel...,D,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...
4,1,Pick the best possible answer: What is the rel...,E,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...


In [86]:
test_ds = Dataset.from_pandas(

    ranking_test[["text"]]

)

test_ds = test_ds.map(

    tokenize,

    batched=True

)

test_ds = test_ds.remove_columns(["text"])

test_ds.set_format("torch")

print(test_ds)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2500
})


In [87]:
predictions = trainer.predict(test_ds)

logits = predictions.predictions

scores = torch.softmax(

    torch.tensor(logits),

    dim=1

)[:, 1].numpy()

print("Prediction Shape:", logits.shape)

print("Score Shape:", scores.shape)

Prediction Shape: (2500, 2)
Score Shape: (2500,)


In [88]:
ranking_test["score"] = scores

ranking_test.head()

,id,question,option_label,option_text,retrieved_context,text,score
0,1,Pick the best possible answer: What is the rel...,A,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...,0.999637
1,1,Pick the best possible answer: What is the rel...,B,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...,0.000110
2,1,Pick the best possible answer: What is the rel...,C,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...,0.000102
3,1,Pick the best possible answer: What is the rel...,D,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...,0.000103
4,1,Pick the best possible answer: What is the rel...,E,"For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...,Retrieved Context: Pick the best possible answ...,0.000107


In [89]:
submission = []

for question_id, group in ranking_test.groupby("id"):

    group = group.sort_values(

        "score",

        ascending=False

    )

    top3 = group["option_label"].tolist()[:3]

    submission.append({

        "id": question_id,

        "Prediction": " ".join(top3)

    })

submission = pd.DataFrame(submission)

submission.head()

,id,Prediction
0,1,A B E
1,2,B E C
2,3,B E D
3,4,E C D
4,5,C A B


In [90]:
submission.to_csv(

    "submission.csv",

    index=False

)

print("Submission Saved Successfully!")

submission.head()

Submission Saved Successfully!


,id,Prediction
0,1,A B E
1,2,B E C
2,3,B E D
3,4,E C D
4,5,C A B


In [91]:
metrics_df = pd.DataFrame({

    "Metric": [

        "Validation Loss",

        "Accuracy",

        "F1 Score",

        "MAP@3"

    ],

    "Value": [

        results["eval_loss"],

        results["eval_accuracy"],

        results["eval_f1"],

        results["eval_map3"]

    ]

})

metrics_df

,Metric,Value
0,Validation Loss,0.116430
1,Accuracy,0.979500
2,F1 Score,0.946405
3,MAP@3,0.991250


In [92]:

wandb.finish()

print("W&B Run Finished Successfully!")

eval/accuracy,▁▃▆▇▇████
eval/f1,▁▅▇▇█████
eval/loss,█▆▃▂▂▁▁▁▁
eval/map3,▁▆▇██████
eval/runtime,▂▂▂▃▃▁▁█▄
eval/samples_per_second,▇▇▇▆▆██▁▅
eval/steps_per_second,▇▇▇▆▆██▁▅
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
+5,...


W&B Run Finished Successfully!


In [94]:

import shutil
import os

MODEL_DIR = "/kaggle/working/bert_rag_model"

ZIP_NAME = "/kaggle/working/bert_rag_model"

if os.path.exists(MODEL_DIR):

    shutil.make_archive(

        ZIP_NAME,

        "zip",

        MODEL_DIR

    )

    print("ZIP created successfully!")

    print(ZIP_NAME + ".zip")

else:

    print("Model folder not found!")

ZIP created successfully!
/kaggle/working/bert_rag_model.zip
